In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-aqc-tensor qiskit-addon-utils quimb rustworkx scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Compilare cuantică aproximativă pentru circuite de evoluție în timp

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Estimare utilizare: Cinci minute pe un procesor Eagle (NOTĂ: Aceasta este doar o estimare. Timpul tău de execuție poate varia.)*
## Context
Acest tutorial demonstrează cum să implementezi **Compilarea Cuantică Aproximativă** folosind rețele tensoriale (AQC-Tensor) cu Qiskit pentru a îmbunătăți performanța circuitelor cuantice. Aplicăm AQC-Tensor în contextul unei evoluții în timp de tip Trotter pentru a reduce adâncimea circuitului, menținând în același timp acuratețea simulării, urmând cadrul Qiskit pentru pregătirea stărilor și optimizare. Vei învăța cum să creezi un circuit ansatz de adâncime redusă dintr-un circuit Trotter inițial, să îl optimizezi cu rețele tensoriale și să îl pregătești pentru execuția pe hardware cuantic.

Obiectivul principal este simularea evoluției în timp pentru un Hamiltonian model cu o adâncime redusă a circuitului. Aceasta se realizează folosind addon-ul Qiskit **AQC-Tensor**, [qiskit-addon-aqc-tensor](https://github.com/Qiskit/qiskit-addon-aqc-tensor), care valorifică rețelele tensoriale, în special stările produs de matrice (MPS), pentru a comprima și optimiza circuitul inițial. Prin ajustări iterative, circuitul ansatz comprimat menține fidelitatea față de circuitul original, rămânând în același timp fezabil pentru hardware-ul cuantic actual. Mai multe detalii pot fi găsite în [documentația](/guides/qiskit-addons-aqc) corespunzătoare, cu un [exemplu simplu](/guides/qiskit-addons-aqc-get-started) pentru a începe.

Compilarea Cuantică Aproximativă este deosebit de avantajoasă în simulările cuantice care depășesc timpii de coerență ai hardware-ului, deoarece permite efectuarea simulărilor complexe mai eficient. Acest tutorial te ghidează prin configurarea fluxului de lucru AQC-Tensor în Qiskit, acoperind inițializarea unui Hamiltonian, generarea circuitelor Trotter și transpilarea circuitului final optimizat pentru un dispozitiv țintă.

## Cerințe
Înainte de a începe acest tutorial, asigură-te că ai instalate următoarele:

* Qiskit SDK v1.0 sau mai nou, cu suport pentru [vizualizare](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.22 sau mai nou (`pip install qiskit-ibm-runtime`)
* Addon-ul Qiskit AQC-Tensor (`pip install 'qiskit-addon-aqc-tensor[aer,quimb-jax]'`)

## Configurare

In [1]:
import numpy as np
import quimb.tensor
import datetime
import matplotlib.pyplot as plt

from scipy.linalg import expm
from scipy.optimize import OptimizeResult, minimize

from qiskit.quantum_info import SparsePauliOp, Pauli
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit import QuantumCircuit
from qiskit.synthesis import SuzukiTrotter

from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_aqc_tensor.ansatz_generation import (
    generate_ansatz_from_circuit,
)
from qiskit_addon_aqc_tensor.objective import MaximizeStateFidelity
from qiskit_addon_aqc_tensor.simulation.quimb import QuimbSimulator
from qiskit_addon_aqc_tensor.simulation import tensornetwork_from_circuit
from qiskit_addon_aqc_tensor.simulation import compute_overlap

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.fake_provider import FakeKyiv

from rustworkx.visualization import graphviz_draw

## Partea I. Exemplu la scară mică

Prima parte a acestui tutorial folosește un exemplu la scară mică cu 10 site-uri pentru a ilustra procesul de mapare a unei probleme de simulare cuantică pe un circuit cuantic executabil. Aici vom explora dinamica unui model XXZ cu 10 site-uri, ceea ce ne permite să construim și să optimizăm un circuit cuantic gestionabil înainte de a scala la sisteme mai mari.

Modelul XXZ este studiat pe scară largă în fizică pentru examinarea interacțiunilor de spin și a proprietăților magnetice. Configurăm Hamiltonianul cu condiții la limită deschise și interacțiuni dependente de site între site-urile vecine de-a lungul lanțului.

### Hamiltonianul modelului și observabila

Hamiltonianul pentru modelul nostru XXZ cu 10 site-uri este definit ca:
$$
\hat{\mathcal{H}}_{XXZ} = \sum_{i=1}^{L-1} J_{i,(i+1)}\left(X_i X_{(i+1)}+Y_i Y_{(i+1)}+ 2\cdot Z_i Z_{(i+1)} \right) \, ,
$$

unde $J_{i,(i+1)}$ este un coeficient aleatoriu corespunzător muchiei $(i, i+1)$, iar $L=10$ este numărul de site-uri.

Prin simularea evoluției acestui sistem cu o adâncime redusă a circuitului, putem obține informații despre utilizarea AQC-Tensor pentru comprimarea și optimizarea circuitelor.

#### Configurarea Hamiltonianului și a observabilei

Înainte de a mapa problema, trebuie să configurăm harta de cuplare, Hamiltonianul și observabila pentru modelul XXZ cu 10 site-uri.

In [2]:
# L is the number of sites in the 1D spin chain
L = 10

# Generate the coupling map
edge_list = [(i - 1, i) for i in range(1, L)]
even_edges = edge_list[::2]
odd_edges = edge_list[1::2]
coupling_map = CouplingMap(edge_list)

# Generate random coefficients for our XXZ Hamiltonian
np.random.seed(0)
Js = np.random.rand(L - 1) + 0.5 * np.ones(L - 1)
hamiltonian = SparsePauliOp(Pauli("I" * L))
for i, edge in enumerate(even_edges + odd_edges):
    hamiltonian += SparsePauliOp.from_sparse_list(
        [
            ("XX", (edge), Js[i] / 2),
            ("YY", (edge), Js[i] / 2),
            ("ZZ", (edge), Js[i]),
        ],
        num_qubits=L,
    )

# Generate a ZZ observable between the two middle qubits
observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)

# Generate an initial Néel state |1010101010⟩
initial_state_circuit = QuantumCircuit(L)
for i in range(L):
    if i % 2:
        initial_state_circuit.x(i)

print("Hamiltonian:", hamiltonian)
print("Observable:", observable)
graphviz_draw(coupling_map.graph, method="circo")

Hamiltonian: SparsePauliOp(['IIIIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII'],
              coeffs=[1.        +0.j, 0.52440675+0.j, 0.52440675+0.j, 1.0488135 +0.j,
 0.60759468+0.j, 0.60759468+0.j, 1.21518937+0.j, 0.55138169+0.j,
 0.55138169+0.j, 1.10276338+0.j, 0.52244159+0.j, 0.52244159+0.j,
 1.04488318+0.j, 0.4618274 +0.j, 0.4618274 +0.j, 0.9236548 +0.j,
 0.57294706+0.j, 0.57294706+0.j, 1.14589411+0.j, 0.46879361+0.j,
 0.46879361+0.j, 0.93758721+0.j, 0.6958865 +0.j, 0.6958865 +0.j,
 1.391773  +0.j, 0.73183138+0.j, 0.73183138+0.j, 1.46366276+0.j])
Observable: SparsePauliOp(['IIIIZZIIII'],
              coeffs=[1.+0.j])


<Image src="../docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/527dbada-1.avif" alt="Output of the previous code cell" />

#### Compute the exact expectation value

For a system of this size, we can compute the exact time-evolved expectation value directly using matrix exponentiation. This serves as our ground truth for evaluating the AQC circuit's accuracy.

In [3]:
aqc_evolution_time = 0.2

# Each baseline Trotter step covers dt = aqc_evolution_time / 3
# The subsequent (uncompressed) step covers 1 additional dt
subsequent_evolution_time = aqc_evolution_time / 3
total_evolution_time = aqc_evolution_time + subsequent_evolution_time

# Compute exact expectation value via matrix exponentiation
H_matrix = hamiltonian.to_matrix()
U_exact = expm(-1j * H_matrix * total_evolution_time)

# Build the initial state vector (Néel state)
initial_state_vec = np.zeros(2**L)
state_idx = sum(2**i for i in range(L) if i % 2)
initial_state_vec[state_idx] = 1.0

# Evolve and compute expectation value
evolved_state = U_exact @ initial_state_vec
obs_matrix = observable.to_matrix()
exact_expval = (evolved_state.conj() @ obs_matrix @ evolved_state).real

print(f"AQC evolution time: {aqc_evolution_time}")
print(f"Subsequent evolution time: {subsequent_evolution_time:.6f}")
print(f"Total evolution time: {total_evolution_time:.6f}")
print(f"Exact expectation value: {exact_expval:.6f}")

AQC evolution time: 0.2
Subsequent evolution time: 0.066667
Total evolution time: 0.266667
Exact expectation value: -0.700899


![Output of the previous code cell](../docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/1ea0e102-23d5-4e6e-8ef8-e82843452b19-1.avif)

Cu Hamiltonianul definit, putem proceda la construirea stării inițiale.

In [4]:
aqc_target_num_trotter_steps = 32

aqc_target_circuit = initial_state_circuit.copy()
aqc_target_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
        time=aqc_evolution_time,
    ),
    inplace=True,
)

### Pasul 1: Maparea intrărilor clasice la o problemă cuantică
Acum că am construit Hamiltonianul, definind interacțiunile spin-spin și câmpurile magnetice externe care caracterizează sistemul, urmăm trei pași principali în fluxul de lucru AQC-Tensor:

1. **Generarea circuitului AQC optimizat**: Folosind Trotterizarea, aproximăm evoluția inițială, care este apoi comprimată pentru a reduce adâncimea circuitului.
2. **Crearea circuitului de evoluție în timp rămas**: Capturăm evoluția pentru timpul rămas dincolo de segmentul inițial.
3. **Combinarea circuitelor**: Îmbinăm circuitul AQC optimizat cu circuitul de evoluție rămas într-un circuit complet de evoluție în timp, gata pentru execuție.

Această abordare creează un ansatz de adâncime redusă pentru evoluția țintă, sprijinind simularea eficientă în limitele hardware-ului cuantic actual.

#### Determinarea porțiunii de evoluție în timp de simulat clasic
Scopul nostru este să simulăm evoluția în timp a Hamiltonianului model definit anterior folosind evoluția Trotter. Pentru a face acest proces eficient pentru hardware-ul cuantic, împărțim evoluția în două segmente:

- **Segmentul inițial**: Această porțiune inițială a evoluției, de la $ t_i = 0.0 $ la $ t_f = 0.2 $, este simulabilă cu MPS și poate fi eficient „compilată" folosind AQC-Tensor. Folosind [addon-ul Qiskit AQC-Tensor](https://github.com/Qiskit/qiskit-addon-aqc-tensor), generăm un circuit comprimat pentru acest segment, denumit `aqc_target_circuit`. Deoarece acest segment va fi simulat pe un simulator de rețele tensoriale, ne putem permite să folosim un număr mai mare de straturi Trotter fără a impacta semnificativ resursele hardware. Setăm `aqc_target_num_trotter_steps = 32` pentru acest segment.

- **Segmentul următor**: Această porțiune rămasă a evoluției, de la $ t = 0.2 $ la $ t = 0.4 $, va fi executată pe hardware cuantic, denumit `subsequent_circuit`. Având în vedere limitările hardware-ului, urmărim să folosim cât mai puține straturi Trotter posibil pentru a menține o adâncime gestionabilă a circuitului. Pentru acest segment, folosim `subsequent_num_trotter_steps = 3`.

#### Alegerea timpului de separare
Alegem $t = 0.2$ ca timp de separare pentru a echilibra simulabilitatea clasică cu fezabilitatea hardware. La începutul evoluției, entanglementul în modelul XXZ rămâne suficient de scăzut pentru ca metodele clasice precum MPS să aproximeze cu precizie.

Când alegi un timp de separare, o regulă utilă este să selectezi un punct în care entanglementul este încă gestionabil clasic, dar captează suficient din evoluție pentru a simplifica porțiunea executată pe hardware. Este posibil să fie necesare încercări repetate pentru a găsi cel mai bun echilibru pentru diferiți Hamiltonieni.

In [5]:
aqc_ansatz_num_trotter_steps = 1

aqc_good_circuit = initial_state_circuit.copy()
aqc_good_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=aqc_ansatz_num_trotter_steps),
        time=aqc_evolution_time,
    ),
    inplace=True,
)

aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(
    aqc_good_circuit
)

# Subsequent circuit: 1 non-compressed Trotter step appended after AQC
subsequent_num_trotter_steps = 1
subsequent_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=subsequent_num_trotter_steps),
    time=subsequent_evolution_time,
)

# Baseline Trotter circuit: 4 Trotter steps over total evolution time, no AQC
baseline_num_trotter_steps = 4
baseline_circuit = initial_state_circuit.copy()
baseline_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=baseline_num_trotter_steps),
        time=total_evolution_time,
    ),
    inplace=True,
)

print(
    f"Target circuit:      depth {aqc_target_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"Baseline circuit:    depth {baseline_circuit.depth(lambda x: x.operation.num_qubits == 2)} ({baseline_num_trotter_steps} Trotter steps, time={total_evolution_time:.4f})"
)
print(
    f"Subsequent circuit:  depth {subsequent_circuit.depth(lambda x: x.operation.num_qubits == 2)} ({subsequent_num_trotter_steps} Trotter step, time={subsequent_evolution_time:.4f})"
)
print(
    f"Ansatz circuit:      depth {aqc_ansatz.depth(lambda x: x.operation.num_qubits == 2)}, with {len(aqc_initial_parameters)} parameters"
)
aqc_ansatz.draw("mpl", fold=-1)

Target circuit:      depth 384
Baseline circuit:    depth 48 (4 Trotter steps, time=0.2667)
Subsequent circuit:  depth 12 (1 Trotter step, time=0.0667)
Ansatz circuit:      depth 3, with 156 parameters


<Image src="../docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/78f2665e-1.avif" alt="Output of the previous code cell" />

#### Set up tensor network simulation and build the target MPS

We use the [quimb](https://github.com/jcmgray/quimb) matrix-product state (MPS) circuit simulator, with JAX providing automatic differentiation for the gradient-based optimization. We then build an MPS representation of the target state and evaluate the starting fidelity between the initial ansatz and the target. As the problem instance is a relatively small example, the starting fidelity starts off quite high.

In [6]:
simulator_settings = QuimbSimulator(
    quimb.tensor.CircuitMPS, autodiff_backend="jax"
)

aqc_target_mps = tensornetwork_from_circuit(
    aqc_target_circuit, simulator_settings
)
print("Target MPS maximum bond dimension:", aqc_target_mps.psi.max_bond())

good_mps = tensornetwork_from_circuit(aqc_good_circuit, simulator_settings)
starting_fidelity = abs(compute_overlap(good_mps, aqc_target_mps)) ** 2
print(f"Starting fidelity: {starting_fidelity:.6f}")

Target MPS maximum bond dimension: 5
Starting fidelity: 0.998246


![Output of the previous code cell](../docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/83039f82-97cb-4613-86c9-a8faf0839a02-0.avif)

Pentru a permite o comparație semnificativă, vom genera două circuite suplimentare:

- **Circuitul de comparație AQC**: Acest circuit evoluează până la `aqc_evolution_time`, dar folosește aceeași durată a pasului Trotter ca `subsequent_circuit`. Servește ca o comparație cu `aqc_target_circuit`, arătând evoluția pe care am observa-o fără utilizarea unui număr crescut de pași Trotter. Ne vom referi la acest circuit ca `aqc_comparison_circuit`.

- **Circuitul de referință**: Acest circuit este folosit ca linie de bază pentru a obține rezultatul exact. Simulează evoluția completă folosind rețele tensoriale pentru a calcula rezultatul exact, oferind o referință pentru evaluarea eficacității AQC-Tensor. Ne vom referi la acest circuit ca `reference_circuit`.

In [7]:
aqc_stopping_fidelity = 1
aqc_max_iterations = 500

stopping_point = 1.0 - aqc_stopping_fidelity
objective = MaximizeStateFidelity(
    aqc_target_mps, aqc_ansatz, simulator_settings
)


def callback(intermediate_result: OptimizeResult):
    fidelity = 1 - intermediate_result.fun
    print(
        f"{datetime.datetime.now()} Intermediate result: Fidelity {fidelity:.8f}"
    )
    if intermediate_result.fun < stopping_point:
        raise StopIteration


result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": aqc_max_iterations},
    callback=callback,
)
if result.status not in (0, 1, 99):
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )

print(f"Done after {result.nit} iterations.")
aqc_final_parameters = result.x

2026-05-18 13:14:49.731596 Intermediate result: Fidelity 0.99952882
2026-05-18 13:14:49.734425 Intermediate result: Fidelity 0.99958531
2026-05-18 13:14:49.737101 Intermediate result: Fidelity 0.99960093
2026-05-18 13:14:49.739813 Intermediate result: Fidelity 0.99961046
2026-05-18 13:14:49.742969 Intermediate result: Fidelity 0.99962560
2026-05-18 13:14:49.745916 Intermediate result: Fidelity 0.99964395
2026-05-18 13:14:49.748615 Intermediate result: Fidelity 0.99968150
2026-05-18 13:14:49.753684 Intermediate result: Fidelity 0.99970569
2026-05-18 13:14:49.756208 Intermediate result: Fidelity 0.99973788
2026-05-18 13:14:49.759067 Intermediate result: Fidelity 0.99975385
2026-05-18 13:14:49.762321 Intermediate result: Fidelity 0.99976458
2026-05-18 13:14:49.765526 Intermediate result: Fidelity 0.99977661
2026-05-18 13:14:49.768496 Intermediate result: Fidelity 0.99978663
2026-05-18 13:14:49.771278 Intermediate result: Fidelity 0.99980236
2026-05-18 13:14:49.773735 Intermediate result: 

#### Assemble the final AQC circuit

With the optimized parameters in hand, we bind them to the ansatz and then append the subsequent (uncompressed) Trotter step. The resulting circuit has the depth of a single compressed Trotter step plus one uncompressed step, but the compressed portion approximates the accuracy of 32 Trotter steps.

In [8]:
aqc_final_circuit = aqc_ansatz.assign_parameters(aqc_final_parameters)
aqc_final_circuit.compose(subsequent_circuit, inplace=True)
aqc_final_circuit.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/e09e40de-0.avif" alt="Output of the previous code cell" />

### Step 2: Optimize problem for quantum hardware execution

For this small-scale example, we use a fake backend (`FakeKyiv`) to simulate hardware execution locally. We transpile both the AQC-optimized circuit (`aqc_final_circuit`) and the baseline Trotter circuit (`baseline_circuit`, four Trotter steps over the full evolution time, no AQC) to the backend's instruction set architecture (ISA), with `optimization_level=3` to further reduce circuit depth.

In [9]:
backend = FakeKyiv()

pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=3
)

# Transpile the AQC-optimized circuit (compressed + subsequent step)
isa_circuit = pass_manager.run(aqc_final_circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
print(
    "AQC circuit depth:",
    isa_circuit.depth(lambda x: x.operation.num_qubits == 2),
)

# Transpile the baseline Trotter circuit (no AQC optimization)
isa_baseline_circuit = pass_manager.run(baseline_circuit)
isa_baseline_observable = observable.apply_layout(isa_baseline_circuit.layout)
print(
    "Baseline Trotter circuit depth:",
    isa_baseline_circuit.depth(lambda x: x.operation.num_qubits == 2),
)

AQC circuit depth: 15
Baseline Trotter circuit depth: 27


#### Generarea unui ansatz și a parametrilor inițiali dintr-un circuit Trotter cu mai puțini pași
Acum că am construit cele patru circuite, să continuăm cu fluxul de lucru AQC-Tensor. Mai întâi, construim un circuit „bun" care are același timp de evoluție ca circuitul țintă, dar cu mai puțini pași Trotter (și astfel mai puține straturi).

Apoi trecem acest circuit „bun" funcției `generate_ansatz_from_circuit` a AQC-Tensor. Această funcție analizează conectivitatea cu doi qubiți a circuitului și returnează două lucruri:

1. Un circuit ansatz general, parametrizat, cu aceeași conectivitate cu doi qubiți ca circuitul de intrare.
2. Parametrii care, introduși în ansatz, produc circuitul de intrare (bun).

În curând vom lua acești parametri și îi vom ajusta iterativ pentru a aduce circuitul ansatz cât mai aproape posibil de MPS-ul țintă.

In [10]:
estimator = Estimator(backend)

# Run both circuits
aqc_result = estimator.run([(isa_circuit, isa_observable)]).result()
baseline_result = estimator.run(
    [(isa_baseline_circuit, isa_baseline_observable)]
).result()

### Step 4: Post-process and return result in desired classical format

We extract the expectation values from both runs and compare them to the exact result. The baseline Trotter circuit shows what we would get without AQC at the same circuit depth, while the AQC circuit demonstrates the improvement from tensor-network optimization.

In [11]:
aqc_expval = aqc_result[0].data.evs.tolist()
baseline_expval = baseline_result[0].data.evs.tolist()

print(f"Exact:              {exact_expval:.4f}")
print(
    f"Baseline Trotter:   {baseline_expval:.4f}, |\u0394| = {np.abs(exact_expval - baseline_expval):.4f}  (depth {isa_baseline_circuit.depth(lambda x: x.operation.num_qubits == 2)}, {baseline_num_trotter_steps} steps)"
)
print(
    f"AQC (3+1):          {aqc_expval:.4f}, |\u0394| = {np.abs(exact_expval - aqc_expval):.4f}  (depth {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}, compressed+subsequent)"
)

Exact:              -0.7009
Baseline Trotter:   -0.5400, |Δ| = 0.1609  (depth 27, 4 steps)
AQC (3+1):          -0.5728, |Δ| = 0.1281  (depth 15, compressed+subsequent)


In [12]:
plt.style.use("seaborn-v0_8")

labels = [
    f"Baseline Trotter\n({baseline_num_trotter_steps} steps, depth {isa_baseline_circuit.depth(lambda x: x.operation.num_qubits == 2)})",
    f"AQC (3+1)\n(depth {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)})",
]
values = [baseline_expval, aqc_expval]
colors = ["tab:orange", "tab:blue"]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, values, color=colors, width=0.5)
plt.axhline(
    y=exact_expval,
    color="tab:green",
    linestyle="--",
    linewidth=2,
    label=f"Exact ({exact_expval:.4f})",
)
plt.ylabel("Expected Value")
plt.title(
    "AQC-Tensor (3 compressed + 1 uncompressed) vs Baseline Trotter (10-site XXZ)"
)
plt.legend()
for bar in bars:
    y_val = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        y_val,
        f"{y_val:.4f}",
        ha="center",
        va="bottom" if y_val >= 0 else "top",
    )
plt.axhline(y=0, color="black", linewidth=0.3)
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/77c39ba8-0.avif" alt="Output of the previous code cell" />

#### Alegerea setărilor pentru simularea rețelei tensoriale
Aici folosim simulatorul de circuit cu stare produs de matrice al lui Quimb, împreună cu jax pentru a furniza gradientul.

In [13]:
# -------------------------Step 1-------------------------

# Define the 50-site spin chain
L = 50
edge_list = [(i - 1, i) for i in range(1, L)]
even_edges = edge_list[::2]
odd_edges = edge_list[1::2]
coupling_map = CouplingMap(edge_list)

# Random XXZ Hamiltonian
np.random.seed(0)
Js = np.random.rand(L - 1) + 0.5 * np.ones(L - 1)
hamiltonian = SparsePauliOp(Pauli("I" * L))
for i, edge in enumerate(even_edges + odd_edges):
    hamiltonian += SparsePauliOp.from_sparse_list(
        [
            ("XX", (edge), Js[i] / 2),
            ("YY", (edge), Js[i] / 2),
            ("ZZ", (edge), Js[i]),
        ],
        num_qubits=L,
    )

observable = SparsePauliOp.from_sparse_list(
    [("ZZ", (L // 2 - 1, L // 2), 1.0)], num_qubits=L
)

# Initial Néel state
initial_state_circuit = QuantumCircuit(L)
for i in range(L):
    if i % 2:
        initial_state_circuit.x(i)

# Time parameters
aqc_evolution_time = 0.2
subsequent_evolution_time = aqc_evolution_time / 3
total_evolution_time = aqc_evolution_time + subsequent_evolution_time

# AQC target circuit (high-accuracy, 32 Trotter steps for AQC portion)
aqc_target_num_trotter_steps = 32

aqc_target_circuit = initial_state_circuit.copy()
aqc_target_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
        time=aqc_evolution_time,
    ),
    inplace=True,
)

# Generate ansatz from 1-step Trotter circuit
aqc_good_circuit = initial_state_circuit.copy()
aqc_good_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=1),
        time=aqc_evolution_time,
    ),
    inplace=True,
)

aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(
    aqc_good_circuit
)

# Subsequent circuit: 1 non-compressed Trotter step
subsequent_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=1),
    time=subsequent_evolution_time,
)

# Baseline Trotter circuit: 4 Trotter steps over total evolution time, no AQC
baseline_num_trotter_steps = 4
baseline_circuit = initial_state_circuit.copy()
baseline_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=baseline_num_trotter_steps),
        time=total_evolution_time,
    ),
    inplace=True,
)
print(
    f"Target circuit:  depth {aqc_target_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"Ansatz circuit:  depth {aqc_ansatz.depth(lambda x: x.operation.num_qubits == 2)}, with {len(aqc_initial_parameters)} parameters"
)
print(
    f"Subsequent circuit: depth {subsequent_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"Baseline circuit:   depth {baseline_circuit.depth(lambda x: x.operation.num_qubits == 2)} ({baseline_num_trotter_steps} steps, time={total_evolution_time:.4f})"
)

# Build target MPS and compute reference expectation value
simulator_settings = QuimbSimulator(
    quimb.tensor.CircuitMPS, autodiff_backend="jax"
)
aqc_target_mps = tensornetwork_from_circuit(
    aqc_target_circuit, simulator_settings
)
print("Target MPS maximum bond dimension:", aqc_target_mps.psi.max_bond())

# For the reference expectation value, we need the full evolution (AQC + subsequent)
# Build a high-accuracy full circuit for MPS reference
full_target_circuit = initial_state_circuit.copy()
full_target_circuit.compose(
    generate_time_evolution_circuit(
        hamiltonian,
        synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
        time=total_evolution_time,
    ),
    inplace=True,
)
full_target_mps = tensornetwork_from_circuit(
    full_target_circuit, simulator_settings
)
exact_expval = full_target_mps.local_expectation(
    quimb.pauli("Z") & quimb.pauli("Z"), (L // 2 - 1, L // 2)
).real.item()
print(f"Reference expectation value (from MPS): {exact_expval:.6f}")

# Optimize ansatz parameters
objective = MaximizeStateFidelity(
    aqc_target_mps, aqc_ansatz, simulator_settings
)


def callback(intermediate_result: OptimizeResult):
    fidelity = 1 - intermediate_result.fun
    print(
        f"{datetime.datetime.now()} Intermediate result: Fidelity {fidelity:.8f}"
    )


result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": 500},
    callback=callback,
)
if result.status not in (0, 1, 99):
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )
print(f"Done after {result.nit} iterations.")

# Assemble the final AQC circuit: optimized ansatz + subsequent Trotter step
aqc_final_circuit = aqc_ansatz.assign_parameters(result.x)
aqc_final_circuit.compose(subsequent_circuit, inplace=True)

# -------------------------Step 2-------------------------

service = QiskitRuntimeService()
backend = service.least_busy(min_num_qubits=127)
print(backend)

pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=3
)
isa_circuit = pass_manager.run(aqc_final_circuit)
isa_observable = observable.apply_layout(isa_circuit.layout)
print(
    "AQC circuit depth:",
    isa_circuit.depth(lambda x: x.operation.num_qubits == 2),
)

# Also transpile the baseline Trotter circuit (4 Trotter steps, no AQC)
isa_baseline_circuit = pass_manager.run(baseline_circuit)
isa_baseline_observable = observable.apply_layout(isa_baseline_circuit.layout)
print(
    "Baseline Trotter circuit depth:",
    isa_baseline_circuit.depth(lambda x: x.operation.num_qubits == 2),
)

# -------------------------Step 3-------------------------

# Submit both circuits in a single job
estimator = Estimator(backend)
estimator.options.environment.job_tags = ["TUT_AQCTE"]

job = estimator.run(
    [
        (isa_circuit, isa_observable),
        (isa_baseline_circuit, isa_baseline_observable),
    ]
)
print("Job ID:", job.job_id())

Target circuit:  depth 385
Ansatz circuit:  depth 7, with 816 parameters
Subsequent circuit: depth 12
Baseline circuit:   depth 49 (4 steps, time=0.2667)
Target MPS maximum bond dimension: 5
Reference expectation value (from MPS): -0.738669
2026-05-18 13:02:11.219150 Intermediate result: Fidelity 0.99795732
2026-05-18 13:02:11.232256 Intermediate result: Fidelity 0.99822481
2026-05-18 13:02:11.245160 Intermediate result: Fidelity 0.99829520
2026-05-18 13:02:11.257765 Intermediate result: Fidelity 0.99832379
2026-05-18 13:02:11.270280 Intermediate result: Fidelity 0.99836416
2026-05-18 13:02:11.284116 Intermediate result: Fidelity 0.99840073
2026-05-18 13:02:11.296856 Intermediate result: Fidelity 0.99846863
2026-05-18 13:02:11.309602 Intermediate result: Fidelity 0.99865244
2026-05-18 13:02:11.322012 Intermediate result: Fidelity 0.99872665
2026-05-18 13:02:11.334195 Intermediate result: Fidelity 0.99892335
2026-05-18 13:02:11.346570 Intermediate result: Fidelity 0.99901045
2026-05-18 

In [15]:
# -------------------------Step 4-------------------------

hw_results = job.result()
aqc_expval = hw_results[0].data.evs.tolist()
baseline_expval = hw_results[1].data.evs.tolist()

print(f"Exact (MPS):        {exact_expval:.4f}")
print(
    f"Baseline Trotter:   {baseline_expval:.4f}, |\u0394| = {np.abs(exact_expval - baseline_expval):.4f}"
)
print(
    f"AQC (3+1):          {aqc_expval:.4f}, |\u0394| = {np.abs(exact_expval - aqc_expval):.4f}"
)

labels = [
    f"Baseline Trotter\n({baseline_num_trotter_steps} steps, depth {isa_baseline_circuit.depth(lambda x: x.operation.num_qubits == 2)})",
    f"AQC (3+1)\n(depth {isa_circuit.depth(lambda x: x.operation.num_qubits == 2)})",
]
values = [baseline_expval, aqc_expval]
colors = ["tab:orange", "tab:blue"]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, values, color=colors, width=0.5)
plt.axhline(
    y=exact_expval,
    color="tab:green",
    linestyle="--",
    linewidth=2,
    label=f"Exact ({exact_expval:.4f})",
)
plt.ylabel("Expected Value")
plt.title(
    "AQC-Tensor (3 compressed + 1 uncompressed) vs Baseline Trotter (50-site XXZ)"
)
plt.legend()
for bar in bars:
    y_val = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        y_val,
        f"{y_val:.4f}",
        ha="center",
        va="bottom" if y_val >= 0 else "top",
    )
plt.axhline(y=0, color="black", linewidth=0.3)
plt.tight_layout()
plt.show()

Exact (MPS):        -0.7387
Baseline Trotter:   -0.5955, |Δ| = 0.1432
AQC (3+1):          -0.6734, |Δ| = 0.0653


<Image src="../docs/images/tutorials/approximate-quantum-compilation-for-time-evolution/extracted-outputs/a4dc23fd-494e-46cb-a8f5-d1cd444b96f4-1.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [AQC-Tensor addon documentation](https://qiskit.github.io/qiskit-addon-aqc-tensor/) — includes the related **unitary AQC** technique, which optimizes parametrized circuits to approximate a target unitary operator rather than a prepared state
- [Error mitigation and suppression techniques](/docs/guides/error-mitigation-and-suppression-techniques)
- [Combine error mitigation techniques](/docs/tutorials/combine-error-mitigation-techniques)
</Admonition>